# NeuraFlix | Khowar OCR Hackathon Demo

One Gradio app with two tabs:

1. **CER Evaluation**: choose a generated sample (PNG + matching TXT) or upload an image and provide its reference text.
2. **Live OCR**: upload a Khowar image and inspect the recognized text without needing a reference.

**Runtime:** Kaggle GPU is recommended. Run the notebook cells from top to bottom. The model is loaded only once.

> Security: do not paste Hugging Face access tokens into notebook cells. If your previous notebook contained a real token, revoke it in Hugging Face settings and create a new token only if needed. This notebook does not require login for a public model repository.


In [ ]:
# Install dependencies
!pip uninstall -y -q torchao
!pip install -q "transformers==4.46.3" "tokenizers==0.20.3" gradio einops addict easydict huggingface_hub


In [ ]:
import os
import re
import glob
import time
import threading
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

import gradio as gr
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

OCR_REPO = "zahidazam714/khowar-ocr-v0.1"

# Update this path if your generated PNG/TXT samples are stored elsewhere.
# Expected pairs: sample_name.png and sample_name.txt
DATASET_DIR = "/kaggle/input/datasets/zahidazam/cer-ocr"

# Inference settings retained from your existing notebook.
BASE_SIZE = 1024
IMAGE_SIZE = 768
CROP_MODE = True
OUTPUT_PATH = "/tmp/neuraflix_ocr_out"

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. In Kaggle, enable Settings > Accelerator > GPU, then rerun.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


In [ ]:
# Load the fine-tuned OCR model once
ocr_tok = AutoTokenizer.from_pretrained(OCR_REPO, trust_remote_code=True)

ocr_model = AutoModel.from_pretrained(
    OCR_REPO,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
).eval().cuda()

print("Model loaded:", OCR_REPO)


In [ ]:
def clean_output(raw: str) -> str:
    # Remove model diagnostic lines while keeping recognized text.
    skip_prefixes = ("=====", "BASE:", "PATCHES:")
    lines = [
        line for line in raw.splitlines()
        if line.strip() and not line.lstrip().startswith(skip_prefixes)
    ]
    return "\n".join(lines).strip()


# Protect stdout capture and model inference from overlapping Gradio requests.
_inference_lock = threading.Lock()

def run_ocr_path(image_path: str):
    buffer = StringIO()
    start = time.perf_counter()

    with _inference_lock:
        with redirect_stdout(buffer):
            ocr_model.infer(
                ocr_tok,
                prompt="<image>\nFree OCR.",
                image_file=image_path,
                output_path=OUTPUT_PATH,
                base_size=BASE_SIZE,
                image_size=IMAGE_SIZE,
                crop_mode=CROP_MODE,
                save_results=False,
            )

    elapsed = time.perf_counter() - start
    raw = buffer.getvalue()
    prediction = clean_output(raw)

    base = re.search(r"BASE:\s+torch\.Size\(\[(.*?)\]\)", raw)
    patches = re.search(r"PATCHES:\s+torch\.Size\(\[(.*?)\]\)", raw)

    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    details = (
        f"Model: DeepSeek-OCR-2 + Khowar LoRA ({OCR_REPO})\n"
        f"Global view tokens: [{base.group(1) if base else '-'}]\n"
        f"Local crop tokens: [{patches.group(1) if patches else '-'}]\n"
        f"Inference time: {elapsed:.2f} seconds on {device_name}"
    )
    return prediction, elapsed, details


def normalize_for_cer(text: str) -> str:
    # Only normalize whitespace. This intentionally does not remove punctuation
    # or alter Khowar/Urdu characters.
    return re.sub(r"\s+", " ", text).strip()


def edit_distance(reference: str, prediction: str) -> int:
    # Levenshtein distance, using O(len(prediction)) memory.
    if len(reference) < len(prediction):
        reference, prediction = prediction, reference
    previous = list(range(len(prediction) + 1))
    for i, char_ref in enumerate(reference, 1):
        current = [i]
        for j, char_pred in enumerate(prediction, 1):
            current.append(min(
                previous[j] + 1,              # deletion
                current[j - 1] + 1,            # insertion
                previous[j - 1] + (char_ref != char_pred),  # substitution
            ))
        previous = current
    return previous[-1]


def calculate_cer(reference: str, prediction: str):
    ref = normalize_for_cer(reference)
    pred = normalize_for_cer(prediction)

    if not ref:
        return None, 0, 0

    errors = edit_distance(ref, pred)
    return errors / len(ref), errors, len(ref)


In [ ]:
# Discover paired sample images and ground-truth text files.
def discover_samples(root: str):
    root_path = Path(root)
    if not root_path.exists():
        print(f"Dataset directory not found: {root}")
        print("You can still use CER Evaluation by uploading an image and entering its reference text.")
        return []

    image_paths = []
    for pattern in ("**/*.png", "**/*.jpg", "**/*.jpeg", "**/*.webp"):
        image_paths.extend(root_path.glob(pattern))

    samples = []
    for image_path in sorted(image_paths):
        txt_path = image_path.with_suffix(".txt")
        if txt_path.exists():
            label = str(image_path.relative_to(root_path))
            samples.append((label, str(image_path), str(txt_path)))

    print(f"Found {len(samples)} image/text pairs in {root}")
    return samples


SAMPLES = discover_samples(DATASET_DIR)
SAMPLE_LOOKUP = {label: (image_path, txt_path) for label, image_path, txt_path in SAMPLES}
SAMPLE_CHOICES = [label for label, _, _ in SAMPLES]


def load_dataset_sample(label):
    if not label or label not in SAMPLE_LOOKUP:
        return None, ""
    image_path, txt_path = SAMPLE_LOOKUP[label]
    with open(txt_path, "r", encoding="utf-8") as f:
        reference = f.read().strip()
    with Image.open(image_path) as im:
        image = im.convert("RGB")
    return image, reference


In [ ]:
def evaluate_sample(sample_label, image, reference_text):
    # A selected dataset sample takes priority, and its paired TXT is authoritative.
    if sample_label and sample_label in SAMPLE_LOOKUP:
        image_path, txt_path = SAMPLE_LOOKUP[sample_label]
        with open(txt_path, "r", encoding="utf-8") as f:
            reference_text = f.read().strip()
    elif image is not None:
        image_path = "/tmp/neuraflix_cer_input.png"
        image.convert("RGB").save(image_path, format="PNG")
    else:
        return "", "Upload an image or choose a dataset sample.", "No CER calculated."

    prediction, elapsed, details = run_ocr_path(image_path)

    if not reference_text or not reference_text.strip():
        metrics = "Ground truth is empty. Enter the reference text to calculate CER."
    else:
        cer, errors, ref_chars = calculate_cer(reference_text, prediction)
        if cer is None:
            metrics = "CER cannot be calculated because the normalized reference is empty."
        else:
            metrics = (
                f"CER: {cer:.2%}\n"
                f"Character edit errors: {errors}\n"
                f"Reference character count (after whitespace normalization): {ref_chars}\n"
                f"Character accuracy estimate (1 - CER): {(1 - cer):.2%}"
            )

    return prediction, details, metrics


def run_live_ocr(image):
    if image is None:
        return "", "Upload a Khowar image first."
    image_path = "/tmp/neuraflix_live_input.png"
    image.convert("RGB").save(image_path, format="PNG")
    prediction, _, details = run_ocr_path(image_path)
    return prediction, details


In [ ]:
# Two-page-style UI using tabs: one model instance shared by both tabs.
custom_css = """
.gradio-container {max-width: 1180px !important; margin: auto !important;}
.output-text textarea {direction: rtl; text-align: right; line-height: 1.9;}
"""

with gr.Blocks(title="NeuraFlix | Khowar OCR", theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.Markdown(
        "# NeuraFlix | Khowar OCR\n"
        "DeepSeek-OCR-2 + Khowar LoRA · Hackathon demonstration"
    )

    with gr.Tab("CER Evaluation"):
        gr.Markdown(
            "Evaluate a rendered sample against its paired ground-truth text. "
            "For an external image, upload it and enter the reference text manually."
        )
        with gr.Row():
            with gr.Column(scale=1):
                sample_dropdown = gr.Dropdown(
                    choices=SAMPLE_CHOICES,
                    value=None,
                    label="Generated dataset sample (optional)",
                    filterable=True,
                )
                cer_image = gr.Image(type="pil", label="Khowar image", height=340)
                reference_box = gr.Textbox(
                    label="Ground-truth reference text",
                    lines=7,
                    rtl=True,
                    placeholder="Select a paired sample or paste the exact reference text here...",
                )
                evaluate_button = gr.Button("Run OCR + Calculate CER", variant="primary")
            with gr.Column(scale=1):
                cer_prediction = gr.Textbox(
                    label="OCR prediction",
                    lines=9,
                    rtl=True,
                    elem_classes=["output-text"],
                )
                cer_details = gr.Textbox(label="Inference details", lines=4)
                cer_metrics = gr.Textbox(label="CER metrics", lines=5)

        sample_dropdown.change(
            fn=load_dataset_sample,
            inputs=sample_dropdown,
            outputs=[cer_image, reference_box],
        )
        evaluate_button.click(
            fn=evaluate_sample,
            inputs=[sample_dropdown, cer_image, reference_box],
            outputs=[cer_prediction, cer_details, cer_metrics],
        )

    with gr.Tab("Live OCR"):
        gr.Markdown("Upload a Khowar scan or screenshot. No reference text is required.")
        with gr.Row():
            with gr.Column(scale=1):
                live_image = gr.Image(type="pil", label="Upload Khowar image", height=420)
                live_button = gr.Button("Extract Khowar Text", variant="primary")
            with gr.Column(scale=1):
                live_prediction = gr.Textbox(
                    label="Recognized text",
                    lines=15,
                    rtl=True,
                    elem_classes=["output-text"],
                )
                live_details = gr.Textbox(label="Inference details", lines=4)
        live_button.click(
            fn=run_live_ocr,
            inputs=live_image,
            outputs=[live_prediction, live_details],
        )

demo.queue(default_concurrency_limit=1).launch(share=True, debug=False)
